In [1]:
# %% [markdown]
# # 08 — Risk Scoring Modeling
#
# ## Objective
# Build a first predictive model for member risk scoring.
#
# Initial target:
# - `high_risk_composite_flag`
#
# Main output:
# - `member_risk_scores.csv`

# %%
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

# %%
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

from src.config import PROCESSED_DIR, OUTPUTS_DIR

# %% [markdown]
# ## Load modeling base

# %%
df = pd.read_csv(PROCESSED_DIR / "member_risk_model_base.csv")
print("df:", df.shape)
display(df.head(3))

# %% [markdown]
# ## Select target

# %%
target_col = "high_risk_composite_flag"
assert target_col in df.columns, f"Target {target_col} not found"

print(df[target_col].value_counts(dropna=False))
print("positive rate:", df[target_col].mean())

# %% [markdown]
# ## Define features

# %%
drop_cols = [
    "member_id",
    "high_cost_flag",
    "very_high_cost_flag",
    "high_utilization_flag",
    "very_high_utilization_flag",
    "risk_composite_points",
    "high_risk_composite_flag",
    "cost_bucket",
]

feature_cols = [c for c in df.columns if c not in drop_cols]
X = df[feature_cols].copy()
y = df[target_col].copy()

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("n feature cols:", len(feature_cols))
print("numeric features:", len(numeric_features))
print("categorical features:", len(categorical_features))

# %% [markdown]
# ## Train / test split

# %%
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

# %% [markdown]
# ## Preprocessing and model

# %%
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=10,
    min_samples_leaf=4,
    random_state=42,
    class_weight="balanced_subsample",
    n_jobs=-1
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

# %% [markdown]
# ## Fit model

# %%
pipeline.fit(X_train, y_train)

# %% [markdown]
# ## Evaluate model

# %%
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, digits=3))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

# %% [markdown]
# ## Score entire portfolio

# %%
df["predicted_risk_probability"] = pipeline.predict_proba(X)[:, 1]
df["predicted_risk_flag"] = (df["predicted_risk_probability"] >= 0.50).astype(int)

df["predicted_risk_decile"] = pd.qcut(
    df["predicted_risk_probability"].rank(method="first"),
    q=10,
    labels=[f"D{i}" for i in range(1, 11)]
)

# business bands
df["predicted_risk_segment"] = pd.cut(
    df["predicted_risk_probability"],
    bins=[-0.01, 0.20, 0.40, 0.60, 0.80, 1.01],
    labels=["very_low", "low", "medium", "high", "very_high"]
)

display(
    df[["member_id", "predicted_risk_probability", "predicted_risk_decile", "predicted_risk_segment"]].head(10)
)

# %% [markdown]
# ## Approximate feature importance

# %%
# recover feature names after preprocessing
ohe = pipeline.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]
cat_feature_names = ohe.get_feature_names_out(categorical_features).tolist()
all_feature_names = numeric_features + cat_feature_names

feature_importance = pd.DataFrame({
    "feature": all_feature_names,
    "importance": pipeline.named_steps["model"].feature_importances_
}).sort_values("importance", ascending=False).reset_index(drop=True)

feature_importance.head(25)

# %% [markdown]
# ## Build final scoring output

# %%
output_cols = [c for c in [
    "member_id",
    "age",
    "sex",
    "region",
    "archetype",
    "plan_type",
    "plan_tier",
    "baseline_risk_score",
    "utilization_propensity",
    "acute_event_propensity",
    "misuse_exposure_propensity",
    "claims_count",
    "claim_cost_sum",
    "high_risk_composite_flag",
    "predicted_risk_probability",
    "predicted_risk_flag",
    "predicted_risk_decile",
    "predicted_risk_segment",
] if c in df.columns]

member_risk_scores = df[output_cols].copy()
print("member_risk_scores:", member_risk_scores.shape)
display(member_risk_scores.head(10))

# %% [markdown]
# ## Save outputs

# %%
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

scores_file = OUTPUTS_DIR / "member_risk_scores.csv"
member_risk_scores.to_csv(scores_file, index=False)

importance_file = OUTPUTS_DIR / "member_risk_feature_importance.csv"
feature_importance.to_csv(importance_file, index=False)

print("Saved:", scores_file)
print("Saved:", importance_file)

# %% [markdown]
# ## Final notes

# %%
notes = [
    "This is the first portfolio-wide predictive risk score.",
    "The model is intentionally simple and practical for a first repository version.",
    "Next step: build fraud / abuse rules and anomaly scoring."
]

for i, note in enumerate(notes, 1):
    print(f"{i}. {note}")

PROJECT_ROOT: C:\Users\dolivares\Desktop\novares-securehealth
df: (4000, 44)


,member_id,age,sex,region,city_tier,socioeconomic_band,employment_status,dependents_n,smoker_flag,alcohol_risk_flag,...,premium_annual,pricing_adequacy_ratio,claims_count,high_cost_flag,very_high_cost_flag,high_utilization_flag,very_high_utilization_flag,risk_composite_points,high_risk_composite_flag,cost_bucket
0,MBR000001,47,M,San Pedro,Urban,Upper-Middle,Self-Employed,0,0,0,...,3496.8,0.851,49.0,0,0,1,1,4,1,low
1,MBR000002,35,F,San Pedro,Metro,Lower-Middle,Employed,0,0,0,...,2384.4,1.285,1.0,0,0,0,0,0,0,low
2,MBR000003,65,M,Asunción,Urban,Middle,Self-Employed,1,0,1,...,5161.2,0.994,24.0,0,0,1,0,3,1,low


high_risk_composite_flag
0    3175
1     825
Name: count, dtype: int64
positive rate: 0.20625
n feature cols: 36
numeric features: 18
categorical features: 18
(3000, 36) (1000, 36) (3000,) (1000,)
              precision    recall  f1-score   support

           0      1.000     0.985     0.992       794
           1      0.945     1.000     0.972       206

    accuracy                          0.988      1000
   macro avg      0.972     0.992     0.982      1000
weighted avg      0.989     0.988     0.988      1000

ROC AUC: 0.9996209434838962
Confusion matrix:
[[782  12]
 [  0 206]]


,member_id,predicted_risk_probability,predicted_risk_decile,predicted_risk_segment
0,MBR000001,0.997812,D10,very_high
1,MBR000002,0.000000,D1,very_low
2,MBR000003,0.979234,D9,very_high
3,MBR000004,0.000000,D1,very_low
4,MBR000005,0.712598,D8,high
5,MBR000006,0.956795,D9,very_high
6,MBR000007,0.000000,D1,very_low
7,MBR000008,0.000000,D1,very_low
8,MBR000009,0.769704,D8,high
9,MBR000010,0.000000,D1,very_low


member_risk_scores: (4000, 17)


,member_id,age,sex,region,archetype,plan_type,plan_tier,baseline_risk_score,utilization_propensity,acute_event_propensity,misuse_exposure_propensity,claims_count,high_risk_composite_flag,predicted_risk_probability,predicted_risk_flag,predicted_risk_decile,predicted_risk_segment
0,MBR000001,47,M,San Pedro,Hyper-Utilizer / Misuse Risk,Managed Review,Mid,0.591,0.847,0.205,0.865,49.0,1,0.997812,1,D10,very_high
1,MBR000002,35,F,San Pedro,Healthy Low User,Essential,Basic,0.156,0.104,0.074,0.085,1.0,0,0.000000,0,D1,very_low
2,MBR000003,65,M,Asunción,Chronic Managed,Chronic Care,Premium,0.728,0.776,0.223,0.056,24.0,1,0.979234,1,D9,very_high
3,MBR000004,38,F,Alto Paraná,Healthy Low User,Essential,Basic,0.150,0.161,0.111,0.093,1.0,0,0.000000,0,D1,very_low
4,MBR000005,47,M,Central,Chronic Managed,Chronic Care,Premium,0.884,0.523,0.237,0.128,15.0,0,0.712598,1,D8,high
5,MBR000006,80,F,Itapúa,Chronic Managed,Chronic Care,Premium,0.809,0.796,0.149,0.091,44.0,1,0.956795,1,D9,very_high
6,MBR000007,30,M,Asunción,Preventive User,Standard,Mid,0.227,0.367,0.090,0.103,3.0,0,0.000000,0,D1,very_low
7,MBR000008,42,F,Central,Preventive User,Standard,Mid,0.202,0.429,0.069,0.103,3.0,0,0.000000,0,D1,very_low
8,MBR000009,52,M,Alto Paraná,Chronic Managed,Chronic Care,Premium,0.638,0.547,0.256,0.085,27.0,0,0.769704,1,D8,high
9,MBR000010,47,M,Central,Preventive User,Standard,Mid,0.251,0.237,0.144,0.087,3.0,0,0.000000,0,D1,very_low


Saved: C:\Users\dolivares\Desktop\novares-securehealth\data\outputs\member_risk_scores.csv
Saved: C:\Users\dolivares\Desktop\novares-securehealth\data\outputs\member_risk_feature_importance.csv
1. This is the first portfolio-wide predictive risk score.
2. The model is intentionally simple and practical for a first repository version.
3. Next step: build fraud / abuse rules and anomaly scoring.
